# 05 Validação — Execução em Lote (F5)

Demonstra e valida a Etapa 5 (F5): execução do pipeline completo (`process_image`, F4.6) em lote sobre 50+ imagens, análise de falsos positivos/negativos (F5.2), ajuste fino de parâmetros baseado nos erros observados (F5.3), exportação da tabela consolidada `results/csv_results.csv` (F5.4) e documentação de limitações (F5.5).

Módulo principal: `stage5_validation.py` (`process_batch`, `analyze_failures`, `export_results_csv`).


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from plant_disease.config import SELECTED_DATA_DIR, RESULTS_DIR
from plant_disease.pipeline import process_image, PipelineConfig
from plant_disease.stage2_leaf_seg import LeafSegmentationConfig
from plant_disease.stage3_lesion import LesionDetectionConfig
from plant_disease.stage5_validation import (
    BatchConfig,
    collect_dataset_paths,
    process_batch,
    analyze_failures,
    export_results_csv,
)

%matplotlib inline

print("✓ Imports OK")

## F5.1 — Coleta do dataset e execução em lote

`collect_dataset_paths` varre `SELECTED_DATA_DIR` e infere a categoria de cada imagem a partir do nome da subpasta (ex.: `tomato_healthy`, `potato_late_blight`), a mesma convenção usada em `04_pipeline.ipynb`.

Critério de pronto: rodar em pelo menos 50 imagens, incluindo saudáveis e doentes, sem que um erro em uma única imagem interrompa o lote.

In [ ]:
dataset_paths = collect_dataset_paths(SELECTED_DATA_DIR)

print(f"Total de imagens encontradas em {SELECTED_DATA_DIR}: {len(dataset_paths)}")

por_categoria = pd.Series([c for _, c in dataset_paths]).value_counts()
print(por_categoria)

assert len(dataset_paths) >= 50, (
    "F5.1 exige pelo menos 50 imagens no lote — verifique SELECTED_DATA_DIR "
    "ou aponte para um diretório com mais amostras."
)
print("\n✓ Pelo menos 50 imagens disponíveis — critério de pronto da F5.1 atendido.")

In [ ]:
import time

# gerar_painel=False (padrão de BatchConfig) — evita criar uma Figure
# matplotlib por imagem, essencial para lotes de 50-300 imagens.
batch_config = BatchConfig()

inicio = time.time()
results = process_batch(dataset_paths, config=batch_config)
duracao = time.time() - inicio

print(f"✓ {len(results)} imagens processadas em {duracao:.1f}s "
      f"({duracao / len(results):.3f}s/imagem)")

n_erros = sum(1 for r in results if r["error"] is not None)
print(f"Erros durante o lote: {n_erros} (o lote NÃO foi interrompido por eles)")

## F5.2 — Análise de falsos positivos e falsos negativos

Usamos a categoria/pasta de origem como rótulo de referência:
- **Falso positivo**: imagem de pasta `*_healthy` que o pipeline classificou com severidade diferente de `Saudavel` (viu doença onde não há).
- **Falso negativo**: imagem de pasta doente que o pipeline classificou como `Saudavel` (não viu a doença).

In [ ]:
falhas = analyze_failures(results)

print(f"Total processado: {falhas['total']}  |  erros: {falhas['n_erros']}  |  avaliados: {falhas['n_avaliados']}")
print(f"Falsos positivos: {falhas['n_false_positives']}  (taxa sobre saudáveis: {falhas['fp_rate']:.1%})")
print(f"Falsos negativos: {falhas['n_false_negatives']}  (taxa sobre doentes: {falhas['fn_rate']:.1%})")

print("\n--- Falsos positivos (saudável classificada como doente) ---")
display(pd.DataFrame(falhas["false_positives"]))

print("\n--- Falsos negativos (doente classificada como Saudavel) ---")
display(pd.DataFrame(falhas["false_negatives"]))

In [ ]:
# Inspeção visual de até 4 falhas de cada tipo, com o painel completo
# (F4.5), para entender visualmente o que gerou o erro de classificação.

def _mostrar_paineis(lista_falhas, titulo, limite=4):
    if not lista_falhas:
        print(f"Nenhum caso de '{titulo}' encontrado.")
        return
    for item in lista_falhas[:limite]:
        path = SELECTED_DATA_DIR / item["category"] / item["image"]
        resultado = process_image(str(path), config=PipelineConfig(gerar_painel=True))
        if resultado["panel"] is not None:
            resultado["panel"].suptitle(
                f"[{titulo}] {item['category']} — {item['image']} — "
                f"{resultado['pct_affected']:.1f}% — {resultado['severity']}",
                fontsize=12, fontweight="bold",
            )
            plt.show()

_mostrar_paineis(falhas["false_positives"], "Falso Positivo")
_mostrar_paineis(falhas["false_negatives"], "Falso Negativo")

## F5.3 — Ajuste fino de parâmetros

Padrões de erro observados tipicamente neste tipo de dataset (ver também F5.5):

- **Falsos positivos** costumam vir de reflexos/tons amareladas naturais da nervura central ou bordas da folha entrando no range `yellow_lower/upper`, e de ruído morfológico residual pequeno.
- **Falsos negativos** costumam vir de lesões com área abaixo de `min_contour_area`, ou tons de lesão fora dos ranges HSV padrão (ex.: lesões muito escuras/muito claras).

Ajustes testados abaixo (documentar o valor final escolhido e o motivo):
- aumentar levemente `s_min`/`v_min` do amarelo de lesão (reduz falso positivo por reflexo);
- aumentar `min_contour_area` (reduz ruído confundido com lesão);
- aumentar `morph_kernel_size` da lesão de (3,3) para (5,5) (consolida manchas pequenas fragmentadas).

In [ ]:
# Configuração ajustada — parâmetros candidatos a partir dos erros
# observados na F5.2. Ajuste os valores abaixo conforme o que for
# observado no seu conjunto de imagens antes de fixar como final.
lesion_config_ajustado = LesionDetectionConfig(
    yellow_lower=(15, 50, 50),   # s_min/v_min um pouco mais altos: reduz reflexo amarelado
    morph_kernel_size=(5, 5),    # consolida fragmentos pequenos de lesão
    min_contour_area=15.0,       # descarta ruído residual menor
)

config_ajustado = BatchConfig(
    pipeline_config=PipelineConfig(
        lesion_config=lesion_config_ajustado,
        gerar_painel=False,
    )
)

results_ajustado = process_batch(dataset_paths, config=config_ajustado)
falhas_ajustado = analyze_failures(results_ajustado)

comparacao = pd.DataFrame([
    {"versao": "original", **{k: falhas[k] for k in
        ("n_false_positives", "n_false_negatives", "fp_rate", "fn_rate")}},
    {"versao": "ajustado", **{k: falhas_ajustado[k] for k in
        ("n_false_positives", "n_false_negatives", "fp_rate", "fn_rate")}},
])
display(comparacao)

print(
    "\nCritério de pronto F5.3: melhora observável nos casos de falha "
    "sem piorar significativamente os saudáveis — compare as taxas acima "
    "antes de adotar `lesion_config_ajustado` como padrão em config.py."
)

## F5.4 — Tabela de resultados consolidada

Exporta `results/csv_results.csv` com as colunas `image, class, leaf_px, lesion_px, pct, severity`, para as 50+ imagens do lote (usamos os resultados da configuração escolhida — original ou ajustada, conforme decisão da F5.3).

In [ ]:
# Escolha explícita da versão final para o CSV (troque para
# `results_ajustado` caso a F5.3 confirme melhora sem regressão).
results_finais = results

sucesso = export_results_csv(results_finais, output_path=RESULTS_DIR / "csv_results.csv")
print(f"CSV exportado com sucesso: {sucesso} -> {RESULTS_DIR / 'csv_results.csv'}")

df_final = pd.read_csv(RESULTS_DIR / "csv_results.csv")
print(f"Linhas no CSV: {len(df_final)}")
display(df_final.head(10))

assert len(df_final) == len(results_finais), "Número de linhas do CSV difere do número de imagens processadas"
assert list(df_final.columns) == ["image", "class", "leaf_px", "lesion_px", "pct", "severity"], (
    "Colunas do CSV não seguem exatamente o especificado no plano (F5.4)"
)
print("\n✓ Critério de pronto da F5.4 atendido: CSV com 50+ imagens, colunas corretas, sem linhas faltando.")

## F5.5 — Limitações e casos extremos observados

Registro para entrar no relatório final (F6.3):

- **Variação de iluminação**: os ranges HSV fixos (F3.1) foram calibrados em condições específicas de luz; fotos mais escuras/claras deslocam os canais S/V e podem gerar falso positivo/negativo. Não há normalização automática de exposição no pipeline atual.
- **Lesões muito pequenas**: `min_contour_area` descarta manchas residuais, mas também pode eliminar lesões reais em estágio inicial (poucos pixels) — trade-off entre ruído e sensibilidade.
- **Tons saudáveis confundidos com doença**: nervuras, bordas secas ou áreas de brilho/reflexo especular podem cair nos ranges de amarelo/marrom mesmo em folhas saudáveis.
- **Dependência de fundo uniforme**: `segment_leaf`/`segment_leaf_otsu` (F2) assumem contraste razoável entre folha e fundo; fundos com cores próximas ao verde da folha ou texturas complexas degradam a segmentação e, por consequência, a métrica de severidade (que depende de `leaf_px` como referência).
- **Ranges HSV manuais (F3.1)**: não generalizam automaticamente entre espécies (tomate vs. batata) ou entre datasets diferentes; qualquer novo conjunto de imagens exige nova rodada de validação visual antes de confiar nos números.
- **Ausência de reamostragem entre classes**: se o dataset tiver muito mais imagens saudáveis que doentes (ou vice-versa), as taxas de FP/FN calculadas aqui podem não refletir bem o desempenho em um cenário balanceado — vale reportar os totais absolutos junto com as taxas.

Essas limitações devem ser citadas explicitamente na discussão do relatório final (F6.3), relacionando cada uma às decisões técnicas tomadas nas etapas F1-F4.